In [2]:
#collect dataset
import cv2
import dlib
import os
import time
import numpy as np

# ====================== KONFIGURASI ======================
DATASET_DIR = "training_faces"
JUMLAH_FOTO = 100
DELAY_FOTO = 0.3

# ====================== INPUT NAMA  ======================
nama = input("Masukkan nama : ").strip().lower().replace(" ", "_")

if nama == "":
    print("Nama tidak boleh kosong.")
    exit()

folder_dosen = os.path.join(DATASET_DIR, nama)

if not os.path.exists(DATASET_DIR):
    os.makedirs(DATASET_DIR)

if not os.path.exists(folder_dosen):
    os.makedirs(folder_dosen)

# ====================== INISIALISASI ======================
detector = dlib.get_frontal_face_detector()
cap = cv2.VideoCapture(1)

if not cap.isOpened():
    print("Kamera tidak dapat dibuka.")
    exit()

print(f"\nMulai mengambil dataset untuk: {nama}")
print("Arahkan wajah ke kamera.")
print("Tekan 'q' untuk keluar.\n")

count = 0
last_capture_time = 0

while True:
    ret, frame = cap.read()

    if not ret or frame is None:
        print("Gagal membaca kamera.")
        break

    # Pastikan frame valid untuk OpenCV dan dlib
    frame = frame.astype(np.uint8)

    # Mirror kamera
    frame = cv2.flip(frame, 1)

    # Konversi BGR ke RGB
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Wajib untuk menghindari error dlib Unsupported image type
    rgb = np.ascontiguousarray(rgb, dtype=np.uint8)

    # Deteksi wajah
    faces = detector(rgb)

    if len(faces) > 0:
        face = max(faces, key=lambda rect: rect.width() * rect.height())

        h, w, _ = frame.shape

        margin = 40
        x1 = max(face.left() - margin, 0)
        y1 = max(face.top() - margin, 0)
        x2 = min(face.right() + margin, w)
        y2 = min(face.bottom() + margin, h)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        current_time = time.time()

        if count < JUMLAH_FOTO and (current_time - last_capture_time) >= DELAY_FOTO:
            wajah = frame[y1:y2, x1:x2]

            if wajah.size != 0:
                wajah = cv2.resize(wajah, (300, 300))

                nama_file = f"{nama}_{count + 1:03d}.jpg"
                path_file = os.path.join(folder_dosen, nama_file)

                cv2.imwrite(path_file, wajah)

                count += 1
                last_capture_time = current_time

                print(f"Foto tersimpan: {path_file}")

    cv2.putText(
        frame,
        f"Nama: {nama}",
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

    cv2.putText(
        frame,
        f"Foto: {count}/{JUMLAH_FOTO}",
        (20, 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

    cv2.putText(
        frame,
        "Tekan q untuk keluar",
        (20, 105),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )

    cv2.imshow("Collect Dataset Wajah", frame)

    if count >= JUMLAH_FOTO:
        print(f"\nDataset untuk {nama} selesai dikumpulkan.")
        break

    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("\nPengambilan dataset dihentikan.")
        break

cap.release()
cv2.destroyAllWindows()


Mulai mengambil dataset untuk: amad
Arahkan wajah ke kamera.
Tekan 'q' untuk keluar.

Foto tersimpan: training_faces\amad\amad_001.jpg
Foto tersimpan: training_faces\amad\amad_002.jpg
Foto tersimpan: training_faces\amad\amad_003.jpg
Foto tersimpan: training_faces\amad\amad_004.jpg
Foto tersimpan: training_faces\amad\amad_005.jpg
Foto tersimpan: training_faces\amad\amad_006.jpg
Foto tersimpan: training_faces\amad\amad_007.jpg
Foto tersimpan: training_faces\amad\amad_008.jpg
Foto tersimpan: training_faces\amad\amad_009.jpg
Foto tersimpan: training_faces\amad\amad_010.jpg
Foto tersimpan: training_faces\amad\amad_011.jpg
Foto tersimpan: training_faces\amad\amad_012.jpg
Foto tersimpan: training_faces\amad\amad_013.jpg
Foto tersimpan: training_faces\amad\amad_014.jpg
Foto tersimpan: training_faces\amad\amad_015.jpg
Foto tersimpan: training_faces\amad\amad_016.jpg
Foto tersimpan: training_faces\amad\amad_017.jpg
Foto tersimpan: training_faces\amad\amad_018.jpg
Foto tersimpan: training_faces\

In [3]:
#train model
import dlib
import cv2
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
import joblib
import os

# ====================== KONFIGURASI ======================
SHAPE_PREDICTOR = "shape_predictor_68_face_landmarks.dat"
FACE_RECOG_MODEL = "dlib_face_recognition_resnet_model_v1.dat"

# Inisialisasi dlib
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(SHAPE_PREDICTOR)
face_rec_model = dlib.face_recognition_model_v1(FACE_RECOG_MODEL)


# ====================== FUNGSI ======================
def get_face_encoding(image, face):
    """Mendapatkan encoding wajah dari gambar dan kotak wajah."""
    shape = predictor(image, face)
    return np.array(face_rec_model.compute_face_descriptor(image, shape))


# ====================== LOAD TRAINING DATA ======================
def load_training_data(training_folder="training_faces"):
    """Memuat dan encode semua wajah dari folder training."""
    encodings = []
    names = []

    if not os.path.exists(training_folder):
        print(f"Folder '{training_folder}' tidak ditemukan!")
        return np.array([]), np.array([])

    print("Memuat data training...")

    for person_name in os.listdir(training_folder):
        person_dir = os.path.join(training_folder, person_name)
        if not os.path.isdir(person_dir):
            continue

        for img_file in os.listdir(person_dir):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(person_dir, img_file)
                image = cv2.imread(img_path)

                if image is None:
                    continue

                rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                faces = detector(rgb)

                for face in faces:
                    encoding = get_face_encoding(rgb, face)
                    encodings.append(encoding)
                    names.append(person_name)
                    print(f"  ✓ {person_name} - {img_file}")
                    break  # Ambil satu wajah per gambar

    if len(encodings) == 0:
        print("Tidak ada data training yang valid!")
        return np.array([]), np.array([])

    print(f"\nTotal {len(encodings)} face encodings dari {len(set(names))} orang telah dimuat.")
    return np.array(encodings), np.array(names)


# ====================== TRAINING ======================
X_train, y_train = load_training_data("training_faces")

if X_train.size > 0:
    # Encode labels
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)

    # Train KNN Classifier
    knn = KNeighborsClassifier(n_neighbors=3, weights='distance', metric='euclidean')
    knn.fit(X_train, y_train_encoded)

    # Simpan model dan encoder
    joblib.dump(knn, 'knn_model_face.pkl')
    joblib.dump(le, 'label_encoder_face.pkl')
    np.save('face_encoding.npy', X_train)

    print("\n✅ Model KNN berhasil di-train dan disimpan!")
    print(f"   - knn_model_face.pkl")
    print(f"   - label_encoder_face.pkl")
    print(f"   - face_encodings.npy")
else:
    print("❌ Gagal training: Tidak ada data training yang valid.")

Memuat data training...
  ✓ amad - amad_001.jpg
  ✓ amad - amad_002.jpg
  ✓ amad - amad_003.jpg
  ✓ amad - amad_004.jpg
  ✓ amad - amad_005.jpg
  ✓ amad - amad_006.jpg
  ✓ amad - amad_007.jpg
  ✓ amad - amad_008.jpg
  ✓ amad - amad_009.jpg
  ✓ amad - amad_010.jpg
  ✓ amad - amad_011.jpg
  ✓ amad - amad_012.jpg
  ✓ amad - amad_013.jpg
  ✓ amad - amad_014.jpg
  ✓ amad - amad_015.jpg
  ✓ amad - amad_016.jpg
  ✓ amad - amad_017.jpg
  ✓ amad - amad_018.jpg
  ✓ amad - amad_019.jpg
  ✓ amad - amad_020.jpg
  ✓ amad - amad_021.jpg
  ✓ amad - amad_022.jpg
  ✓ amad - amad_023.jpg
  ✓ amad - amad_024.jpg
  ✓ amad - amad_025.jpg
  ✓ amad - amad_026.jpg
  ✓ amad - amad_027.jpg
  ✓ amad - amad_028.jpg
  ✓ amad - amad_030.jpg
  ✓ amad - amad_031.jpg
  ✓ amad - amad_032.jpg
  ✓ amad - amad_033.jpg
  ✓ amad - amad_034.jpg
  ✓ amad - amad_035.jpg
  ✓ amad - amad_036.jpg
  ✓ amad - amad_037.jpg
  ✓ amad - amad_038.jpg
  ✓ amad - amad_039.jpg
  ✓ amad - amad_040.jpg
  ✓ amad - amad_041.jpg
  ✓ amad - amad_